# 🏥 Sistema Básico de Apoio ao Diagnóstico Médico

Este notebook demonstra o funcionamento de um sistema simples de extração de sintomas e sugestão de diagnósticos.

### Estrutura do projeto
| Arquivo | Descrição |
|---|---|
| `sintomas_pacientes.txt` | 10 frases simulando relatos de pacientes |
| `mapa_conhecimento.csv` | Mapa de sintomas → doenças (30 entradas) |
| `diagnostico_sintomas.ipynb` | Este notebook com a lógica de análise |

> ⚠️ **Aviso:** Este sistema é estritamente educacional e **não substitui** avaliação médica profissional.

---
## Passo 1 — Importações e funções auxiliares

In [ ]:
import csv
import re
from collections import defaultdict

print('✅ Bibliotecas carregadas com sucesso!')

---
## Passo 2 — Carregar o mapa de conhecimento (CSV)

O arquivo `mapa_conhecimento.csv` contém associações entre sintomas e doenças.
Cada linha tem 3 sintomas e 1 doença associada.

In [ ]:
def carregar_mapa(caminho_csv):
    mapa = []
    with open(caminho_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sintomas = [
                row['Sintoma 1'].strip().lower(),
                row['Sintoma 2'].strip().lower(),
                row['Sintoma 3'].strip().lower(),
            ]
            doenca = row['Doenca Associada'].strip()
            mapa.append((sintomas, doenca))
    return mapa

mapa = carregar_mapa('mapa_conhecimento.csv')

print(f'📋 Mapa carregado com {len(mapa)} entradas.\n')
print(f'{"SINTOMAS":<55} {"DOENÇA"}')
print('-' * 85)
for sintomas, doenca in mapa:
    linha_sintomas = ' | '.join(sintomas)
    print(f'{linha_sintomas:<55} {doenca}')

---
## Passo 3 — Carregar as frases dos pacientes (TXT)

O arquivo `sintomas_pacientes.txt` contém 10 relatos simulados de pacientes,
cada um descrevendo o que sente, quando começou e como afeta sua rotina.

In [ ]:
def carregar_frases(caminho_txt):
    frases = []
    with open(caminho_txt, encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                linha = re.sub(r'^\d+\.\s*', '', linha)
                frases.append(linha)
    return frases

frases = carregar_frases('sintomas_pacientes.txt')

print(f'📄 {len(frases)} frases carregadas:\n')
for i, frase in enumerate(frases, 1):
    print(f'  Paciente {i:02d}: {frase[:100]}...')

---
## Passo 4 — Lógica de extração de sintomas e diagnóstico

Para cada frase, o sistema busca correspondências com os sintomas do mapa.
A doença com mais correspondências é sugerida como diagnóstico principal.

In [ ]:
def analisar_frase(frase, mapa):
    frase_lower = frase.lower()
    pontuacao = defaultdict(int)
    sintomas_encontrados = defaultdict(list)

    for sintomas, doenca in mapa:
        for sintoma in sintomas:
            if sintoma and sintoma in frase_lower:
                pontuacao[doenca] += 1
                sintomas_encontrados[doenca].append(sintoma)

    if not pontuacao:
        return [], {}

    ranking = sorted(pontuacao.items(), key=lambda x: x[1], reverse=True)
    return ranking, sintomas_encontrados

print('✅ Função de análise definida com sucesso!')

---
## Passo 5 — Executar a análise para todos os pacientes

In [ ]:
for i, frase in enumerate(frases, start=1):
    print('=' * 70)
    print(f'  PACIENTE {i:02d}')
    print('=' * 70)
    print(f'  Relato: {frase}\n')

    ranking, sintomas_enc = analisar_frase(frase, mapa)

    if not ranking:
        print('  ⚠  Nenhum sintoma reconhecido no mapa de conhecimento.\n')
    else:
        principal, pts = ranking[0]
        encontrados = list(set(sintomas_enc[principal]))

        print(f'  ✔  Diagnostico sugerido  : {principal}')
        print(f'     Sintomas identificados : {", ".join(encontrados)}')
        print(f'     Confianca              : {pts} correspondencia(s)')

        if len(ranking) > 1:
            alternativas = [d for d, _ in ranking[1:3]]
            print(f'     Alternativas           : {", ".join(alternativas)}')
    print()

---
## Passo 6 — Análise individual (teste com uma frase personalizada)

Você pode testar o sistema com qualquer frase. Edite a variável `frase_teste` abaixo:

In [ ]:
# ✏️ Edite esta frase para testar com qualquer relato!
frase_teste = "Há dois dias estou sentindo uma dor forte no peito e uma sensação de aperto no tórax que irradia para o braço esquerdo, especialmente quando faço qualquer esforço físico. Também sinto falta de ar e suor frio" \
"

print(f'Frase analisada: "{frase_teste}"\n')

ranking, sintomas_enc = analisar_frase(frase_teste, mapa)

if not ranking:
    print('Nenhum sintoma reconhecido.')
else:
    print(f'{'Doença':<40} {'Correspondencias':<18} {'Sintomas encontrados'}')
    print('-' * 85)
    for doenca, pts in ranking:
        enc = ', '.join(set(sintomas_enc[doenca]))
        print(f'{doenca:<40} {pts:<18} {enc}')

---
## Conclusão

Este sistema demonstra como é possível estruturar um **mapa de conhecimento** simples para auxiliar na identificação de possíveis diagnósticos a partir de relatos textuais de pacientes.

### Como o sistema funciona:
1. **Lê** as frases dos pacientes em linguagem natural
2. **Busca** palavras-chave de sintomas dentro de cada frase
3. **Pontua** cada doença conforme o número de sintomas encontrados
4. **Sugere** o diagnóstico com maior pontuação como principal

### Possíveis melhorias futuras:
- Uso de **PLN (Processamento de Linguagem Natural)** para reconhecimento semântico
- Integração com **modelos de Machine Learning** para maior precisão
- Expansão do mapa de conhecimento com mais doenças e sintomas

> ⚠️ Este projeto é estritamente **educacional** e não deve ser usado para fins médicos reais.